In [0]:
import requests
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType
from datetime import datetime, timedelta


# Pipeline Silver to Gold - Camada Analítica

## Objetivo
Este notebook realiza a transformação dos dados da camada **Silver** (dados limpos e normalizados) para a camada **Gold** (dados agregados e otimizados para análise).

## Tabelas Gold Criadas
1. **fat_vendas_comercial**: Fatos de vendas agregados por ano, mês, produto e categoria
2. **fat_avaliacoes_clientes**: Fatos de avaliações agregados por vendedor, categoria e estado

## Estratégia de Modelagem
- **Agregações temporais**: Agrupamentos por ano/mês para análises de tendências
- **Métricas de negócio**: Receita total, ticket médio, satisfação do cliente
- **Conversão cambial**: Valores em BRL e USD para análises internacionais

In [0]:
Catalogo_Name = "visagio"
DB_Gold = "gold"
DB_Silver = "silver"
DB_Landing = "/Volumes/visagio/landing/atividade_rocketlab/"

# Tabela Gold: fat_vendas_comercial

## Objetivo
Criar uma tabela fato agregada com métricas de vendas por período, produto e categoria para análises comerciais e dashboards executivos.

## Regras de Negócio Aplicadas

### 1. Joins de Origem (Silver)
- **fat_itens_pedidos**: Base de itens vendidos com preços
- **fat_pedido_total**: Datas dos pedidos e status
- **dim_produtos**: Nomes e categorias de produtos
- **dim_cotacao_dolar**: Cotação diária para conversão cambial

### 2. Transformações Aplicadas

#### 2.1 Conversão Cambial
- **Regra**: Converter preços de BRL para USD usando cotação da data do pedido
- **Cálculo**: `preco_usd = preco_BRL / cotacao_compra`
- **Justificativa**: Permitir análises de receita em moeda internacional

#### 2.2 Dimensões Temporais
- **Regra**: Extrair ano e mês da data do pedido
- **Campos criados**: `ano_venda`, `mes_venda`
- **Justificativa**: Facilitar agregações temporais e tendências sazonais

#### 2.3 Tratamento de Nulos
- **Regra**: Substituir categoria nula por "Sem Categoria"
- **Justificativa**: Evitar perda de registros em agregações e facilitar visualizações

### 3. Métricas Calculadas

| Métrica | Fórmula | Justificativa |
|---------|---------|---------------|
| **total_pedidos** | `countDistinct(id_pedido)` | Número único de pedidos (um pedido pode ter vários itens) |
| **qtd_itens_vendidos** | `count(id_item)` | Volume total de itens comercializados |
| **receita_total_brl** | `sum(preco_BRL)` | Faturamento em moeda local |
| **receita_total_usd** | `sum(preco_usd)` | Faturamento em moeda internacional |
| **ticket_medio_brl** | `sum(preco_BRL) / countDistinct(id_pedido)` | Valor médio por pedido para análise de rentabilidade |

### 4. Agregação Final
- **Nível de granularidade**: Ano + Mês + Produto + Categoria
- **Ordenação**: Cronológica (ano, mês)
- **Uso esperado**: Dashboards de vendas, análise de tendências, previsão de demanda

### 5. Otimizações Aplicadas
- **Formato**: Delta Lake com `overwriteSchema`
- **Modo**: Overwrite completo para garantir consistência
- **Precisão**: Arredondamento de 2 casas decimais para valores monetários

In [0]:
def create_fat_vendas_comercial():
    try:
        target_table = f"{Catalogo_Name}.{DB_Gold}.fat_vendas_comercial"

        df_itens    = spark.table(f"{Catalogo_Name}.{DB_Silver}.fat_itens_pedidos")
        df_pedidos  = spark.table(f"{Catalogo_Name}.{DB_Silver}.fat_pedido_total")
        df_produtos = spark.table(f"{Catalogo_Name}.{DB_Silver}.dim_produtos")
        df_cotacao  = spark.table(f"{Catalogo_Name}.{DB_Silver}.dim_cotacao_dolar")

        df_itens_produtos = (
            df_itens
            .join(df_produtos, on="id_produto", how="left")
        )

        df_completo = (
            df_itens_produtos
            .join(df_pedidos, on="id_pedido", how="left")
            .join(
                df_cotacao,
                on=F.to_date(df_pedidos["data_pedido"]) == F.to_date(df_cotacao["data_cotacao"], "dd-MM-yyyy"),
                how="left"
            )
            .withColumn("preco_usd",
                F.round(F.col("preco_BRL") / F.col("cotacao_compra"), 2)
            )
            .withColumn("ano_venda", F.year("data_pedido"))
            .withColumn("mes_venda", F.month("data_pedido"))
            .drop(df_cotacao["data_cotacao"])
        )

        df_gold = (
            df_completo
            .withColumn("categoria_produto",
                F.when(F.col("categoria_produto").isNull(), "Sem Categoria")
                .otherwise(F.col("categoria_produto"))
            )
            .groupBy("ano_venda", "mes_venda","nome_produto", "categoria_produto")
            .agg(
                F.countDistinct("id_pedido").alias("total_pedidos"),
                F.count("id_item").alias("qtd_itens_vendidos"),
                F.round(F.sum("preco_BRL"), 2).alias("receita_total_brl"),
                F.round(F.sum("preco_usd"), 2).alias("receita_total_usd"),
                F.round(F.sum("preco_BRL") / F.countDistinct("id_pedido"), 2).alias("ticket_medio_brl"),
            )
            .select(
                F.col("ano_venda"),
                F.col("mes_venda"),
                F.col("nome_produto"),
                F.col("categoria_produto"),
                F.col("total_pedidos"),
                F.col("qtd_itens_vendidos"),
                F.col("receita_total_brl"),
                F.col("receita_total_usd"),
                F.col("ticket_medio_brl"),
            )
            .orderBy("ano_venda", "mes_venda")
        )

        df_gold.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"✅ Tabela {target_table} atualizada com sucesso! ({df_gold.count()} registros)\n")

    except Exception as e:
        print(f"❌ Erro ao criar gold.fat_vendas_comercial: {str(e)}")

create_fat_vendas_comercial()

✅ Tabela visagio.gold.fat_vendas_comercial atualizada com sucesso! (30467 registros)



# Análise Exploratória: Top Produtos

## Objetivo
Identificar produtos com melhor e pior desempenho de vendas para apoiar decisões estratégicas de gestão de portfólio.

## Regras de Negócio

### Top 5 Produtos MAIS Vendidos
- **Métrica**: Soma da quantidade de itens vendidos (`qtd_itens_vendidos`)
- **Agregação**: Por `nome_produto`
- **Ordenação**: Decrescente (maior para menor)
- **Uso**: Identificar produtos de alto giro para priorizar estoque e promoções

### Top 5 Produtos MENOS Vendidos
- **Métrica**: Soma da quantidade de itens vendidos (`qtd_itens_vendidos`)
- **Agregação**: Por `nome_produto`
- **Ordenação**: Crescente (menor para maior)
- **Uso**: Identificar produtos de baixo giro para revisar preços, descontinuar ou fazer liquidação

## Insights Esperados
- Produtos campeões de vendas (best-sellers)
- Produtos com baixa demanda (candidatos a desconto ou descontinuação)
- Padrões de preferência do consumidor

In [0]:
df_gold = spark.table(f"{Catalogo_Name}.{DB_Gold}.fat_vendas_comercial")

print("🏆 Top 5 Produtos MAIS Vendidos")
display(
    df_gold
    .groupBy("nome_produto")
    .agg(F.sum("qtd_itens_vendidos").alias("qtd_vendida"))
    .orderBy(F.desc("qtd_vendida"))
    .limit(5)
)

print("📉 Top 5 Produtos MENOS Vendidos")
display(
    df_gold
    .groupBy("nome_produto")
    .agg(F.sum("qtd_itens_vendidos").alias("qtd_vendida"))
    .orderBy(F.asc("qtd_vendida"))
    .limit(5)
)

🏆 Top 5 Produtos MAIS Vendidos


nome_produto,qtd_vendida
Secador de Cabelo,1076
Toalha de Banho,919
Acessório Padrão,919
Protetor Solar,917
Produto Genérico,851


📉 Top 5 Produtos MENOS Vendidos


nome_produto,qtd_vendida
Aquarela Ultra,1
Teclado Musical Max,1
Luminária Criativa Branco,1
Livro de Receitas Amarelo,1
Cerveja Artesanal Luxo,1


# Tabela Gold: fat_avaliacoes_clientes

## Objetivo
Agregar avaliações de clientes por categoria de produto, vendedor e estado para análise de satisfação e qualidade de atendimento.

## Regras de Negócio Aplicadas

### 1. Joins de Origem (Silver)
- **fat_avaliacoes_pedidos**: Base de avaliações com notas e comentários
- **fat_itens_pedidos**: Ligação entre pedidos e produtos
- **dim_produtos**: Categorias de produtos
- **dim_vendedores**: Identificação e localização dos vendedores

### 2. Transformações Aplicadas

#### 2.1 Tratamento de Nulos
- **Regra**: Substituir categoria nula por "Sem Categoria"
- **Justificativa**: Manter todas as avaliações na análise, mesmo sem categoria definida

#### 2.2 Classificação de Avaliações
- **Avaliações Positivas**: Nota >= 4 (clientes satisfeitos)
- **Avaliações Negativas**: Nota <= 2 (clientes insatisfeitos)
- **Justificativa**: Segmentar feedback para identificar pontos fortes e fracos

### 3. Métricas Calculadas

| Métrica | Fórmula | Justificativa |
|---------|---------|---------------|
| **total_avaliacoes** | `count(id_avaliacao)` | Volume de feedback recebido |
| **avaliacao_media** | `avg(nota_avaliacao)` | Satisfação geral do cliente |
| **total_avaliacoes_positivas** | `sum(nota >= 4)` | Quantidade de clientes satisfeitos |
| **total_avaliacoes_negativas** | `sum(nota <= 2)` | Quantidade de clientes insatisfeitos |
| **percentual_satisfacao** | `(positivas / total) * 100` | Taxa de satisfação em % para benchmarking |

### 4. Agregação Final
- **Nível de granularidade**: Categoria + Vendedor + Estado
- **Dimensão geográfica**: Estado do vendedor para análises regionais
- **Uso esperado**: 
  - Ranking de vendedores por satisfação
  - Identificação de produtos problemáticos
  - Análise de performance regional

### 5. Otimizações Aplicadas
- **Formato**: Delta Lake com `overwriteSchema`
- **Modo**: Overwrite completo
- **Precisão**: 2 casas decimais para percentuais e médias

In [0]:
def create_fat_avaliacoes_clientes():
    try:
        target_table = f"{Catalogo_Name}.{DB_Gold}.fat_avaliacoes_clientes"

        df_avaliacoes = spark.table(f"{Catalogo_Name}.{DB_Silver}.fat_avaliacoes_pedidos")
        df_itens      = spark.table(f"{Catalogo_Name}.{DB_Silver}.fat_itens_pedidos")
        df_produtos   = spark.table(f"{Catalogo_Name}.{DB_Silver}.dim_produtos")
        df_vendedores = spark.table(f"{Catalogo_Name}.{DB_Silver}.dim_vendedores")

        df_completo = (
            df_avaliacoes
            .join(df_itens,      on="id_pedido",  how="left")
            .join(df_produtos,   on="id_produto", how="left")
            .join(df_vendedores, on="id_vendedor", how="left")
        )

        df_gold = (
            df_completo
            .withColumn("categoria_produto",
                F.when(F.col("categoria_produto").isNull(), "Sem Categoria")
                .otherwise(F.col("categoria_produto"))
            )
            .groupBy(
                F.col("categoria_produto"),
                F.col("id_vendedor"),
                F.col("nome_vendedor"),
                F.col("estado")
            )
            .agg(
                F.count("id_avaliacao").alias("total_avaliacoes"),
                F.round(F.avg("nota_avaliacao"),2).alias("avaliacao_media"),
                F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)).alias("total_avaliacoes_positivas"),
                F.sum(F.when(F.col("nota_avaliacao") <= 2, 1).otherwise(0)).alias("total_avaliacoes_negativas"),
                F.round(
                F.sum(F.when(F.col("nota_avaliacao") >= 4, 1).otherwise(0)) /
                    F.count("id_avaliacao") * 100, 2
                ).alias("percentual_satisfacao"),
            )
            .select(
                F.col("categoria_produto"),
                F.col("nome_vendedor"),
                F.col("id_vendedor"),
                F.col("estado").alias("estado_vendedor"),
                F.col("total_avaliacoes"),
                F.col("avaliacao_media"),
                F.col("total_avaliacoes_positivas"),
                F.col("total_avaliacoes_negativas"),
                F.col("percentual_satisfacao"),
            )
        )
        df_gold.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(target_table)

        print(f"✅ Tabela {target_table} atualizada com sucesso! ({df_gold.count()} registros)\n")

    except Exception as e:
        print(f"❌ Erro ao criar gold.fat_avaliacoes_clientes: {str(e)}")


create_fat_avaliacoes_clientes()    

✅ Tabela visagio.gold.fat_avaliacoes_clientes atualizada com sucesso! (6591 registros)



# Análise Exploratória: Ranking de Avaliações

## Objetivo
Identificar produtos e vendedores com melhor e pior desempenho em avaliações de clientes para ações de melhoria e reconhecimento.

## Regras de Negócio

### 1. Ranking por Categoria de Produto

#### Produto MAIS Bem Avaliado
- **Métrica primária**: Média das avaliações (`avaliacao_media`)
- **Métrica secundária**: Total de avaliações (para desempate)
- **Ordenação**: Decrescente (maior nota)
- **Uso**: Identificar categorias de maior satisfação para investir e expandir

#### Produto MENOS Bem Avaliado
- **Métrica primária**: Média das avaliações (`avaliacao_media`)
- **Métrica secundária**: Total de avaliações (para desempate)
- **Ordenação**: Crescente (menor nota)
- **Uso**: Identificar categorias problemáticas para ações corretivas

### 2. Ranking por Vendedor

#### Vendedor MAIS Bem Avaliado
- **Métrica primária**: Média das avaliações (`avaliacao_media`)
- **Métrica secundária**: Total de avaliações (para desempate)
- **Ordenação**: Decrescente (maior nota)
- **Uso**: Reconhecer vendedores de excelência e replicar boas práticas

#### Vendedor MENOS Bem Avaliado
- **Métrica primária**: Média das avaliações (`avaliacao_media`)
- **Métrica secundária**: Total de avaliações (para desempate)
- **Ordenação**: Crescente (menor nota)
- **Uso**: Identificar vendedores que precisam de treinamento ou suporte

## Insights Esperados
- Identificação de best practices (vendedores/produtos top)
- Plano de ação para categorias/vendedores com baixa satisfação
- Base para sistema de reconhecimento e bonificação
- Identificação de necessidades de treinamento

## Critério de Desempate
Quando há empate na avaliação média, o desempate é feito pelo **total de avaliações** (maior volume = mais representativo).

In [0]:
df_aval = spark.table(f"{Catalogo_Name}.{DB_Gold}.fat_avaliacoes_clientes")

# Agrega por produto
df_produtos_rank = (
    df_aval
    .groupBy("categoria_produto")
    .agg(
        F.round(F.avg("avaliacao_media"), 2).alias("avaliacao_media"),
        F.sum("total_avaliacoes").alias("total_avaliacoes")
    )
)

# Agrega por vendedor
df_vendedores_rank = (
    df_aval
    .groupBy("nome_vendedor", "estado_vendedor")
    .agg(
        F.round(F.avg("avaliacao_media"), 2).alias("avaliacao_media"),
        F.sum("total_avaliacoes").alias("total_avaliacoes")
    )
)

# 1. Produto MAIS bem avaliado
print("🏆 Produto MAIS Bem Avaliado")
display(
    df_produtos_rank
    .orderBy(F.desc("avaliacao_media"), F.desc("total_avaliacoes"))
    .limit(1)
)

# 2. Produto MENOS bem avaliado
print("📉 Produto MENOS Bem Avaliado")
display(
    df_produtos_rank
    .orderBy(F.asc("avaliacao_media"), F.desc("total_avaliacoes"))
    .limit(1)
)

# 3. Vendedor MAIS bem avaliado
print("🏆 Vendedor MAIS Bem Avaliado")
display(
    df_vendedores_rank
    .orderBy(F.desc("avaliacao_media"), F.desc("total_avaliacoes"))
    .limit(1)
)

# 4. Vendedor MENOS bem avaliado
print("📉 Vendedor MENOS Bem Avaliado")
display(
    df_vendedores_rank
    .orderBy(F.asc("avaliacao_media"), F.desc("total_avaliacoes"))
    .limit(1)
)

🏆 Produto MAIS Bem Avaliado


categoria_produto,avaliacao_media,total_avaliacoes
cds_dvds_musicais,4.64,14


📉 Produto MENOS Bem Avaliado


categoria_produto,avaliacao_media,total_avaliacoes
seguros_e_servicos,2.5,2


🏆 Vendedor MAIS Bem Avaliado


nome_vendedor,estado_vendedor,avaliacao_media,total_avaliacoes
Luiz Otávio Abreu,PR,5.0,34


📉 Vendedor MENOS Bem Avaliado


nome_vendedor,estado_vendedor,avaliacao_media,total_avaliacoes
Sra. Fernanda Santos,SP,1.0,8
